In [1]:
from tinydas.trainer import Trainer
from tinydas.dataset import DataSet
from tinydas.dataloader import DataLoader
from tinydas.utils import parse_args, get_config, get_gpus, seed_all
from tinydas.models.ae import AE

from tqdm import trange
from tinygrad import nn, TinyJit, GlobalCounters, Device
from tinygrad.nn import Tensor

In [2]:
epochs = 10
bs = 5
seed = 1234
n = 10
lr = 0.01

In [3]:
config = get_config("./config/ae.yaml")

seed_all(seed)

In [4]:
GPUS = get_gpus()

print(f"Training on {GPUS}")

Training on ['CUDA:0', 'CUDA:1']


In [5]:
ds = DataSet("./2023", n=n)

In [6]:
dl = DataLoader(dataset=ds, batch_size=bs)

In [7]:
class Model:
    def __init__(self):
        self.layers = [
            nn.Linear(2137 * 7500, 64), Tensor.relu,
            nn.Linear(64, 2137 * 7500), Tensor.sigmoid
        ]

    def __call__(self, x: Tensor) -> Tensor:
        return x.sequential(self.layers)

In [8]:
model = Model()

In [9]:
for _, x in nn.state.get_state_dict(model).items(): x.to_(GPUS)
opt = nn.optim.Adam(nn.state.get_parameters(model), lr=lr)

In [10]:
@TinyJit
def train_step() -> Tensor:
    with Tensor.train():  
        opt.zero_grad()
        
        for d, _ in dl:
            X = d.shard_(GPUS, axis=0).reshape(-1, 2137 * 7500)
            loss = model(X).sub(X).square().mean().backward()
            #loss.backward()
        opt.step()

        return loss

In [ ]:
losses = [0.0] * epochs
Tensor.training = True
for epoch in (t := trange(epochs)):
    GlobalCounters.reset()
    loss = train_step()
    t.set_description(f"Epoch: {epoch + 1} | loss: {loss.item():.2f}")
    losses[epoch] = loss.item()
    

  0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.plot(losses)
plt.show()